# Debruiteurs Deep

In [84]:
from denoiser import pretrained #DEMUCS
from denoiser import demucs
import sys
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from denoiser.dsp import convert_audio
import torch
import denoiser

sys.path.append("../utils")
import noise
from tools import listen

In [85]:
#Import des fichiers sonores et verification de l'echantillonage
son1,fs1=sf.read('../../data/son1.wav')
print('Echantillonage 1 :',fs1)
son2,fs2=sf.read('../../data/son2.wav')
print('Echantillonage 2 :',fs2)
son3,fs3=sf.read('../../data/son3.wav')
print('Echantillonage 3 :',fs3)

Echantillonage 1 : 44100
Echantillonage 2 : 44100
Echantillonage 3 : 44100


### DEMUCS denoiser

In [86]:
isnr=1
son1_b,s1=noise.add_white_noise(son1,fs1,isnr)
listen(son1_b,44100)
son2_b,s2=noise.add_white_noise(son2,fs2,isnr)
listen(son2_b,44100)

In [87]:
help(denoiser)

Help on package denoiser:

NAME
    denoiser

DESCRIPTION
    # Copyright (c) Facebook, Inc. and its affiliates.
    # All rights reserved.
    #
    # This source code is licensed under the license found in the
    # LICENSE file in the root directory of this source tree.

PACKAGE CONTENTS
    audio
    augment
    data
    demucs
    distrib
    dsp
    enhance
    evaluate
    executor
    live
    pretrained
    resample
    solver
    stft_loss
    utils

FILE
    c:\users\alexa\.local\share\mamba\envs\audio-denoising\lib\site-packages\denoiser\__init__.py




In [88]:
help(pretrained)

Help on module denoiser.pretrained in denoiser:

NAME
    denoiser.pretrained

DESCRIPTION
    # Copyright (c) Facebook, Inc. and its affiliates.
    # All rights reserved.
    #
    # This source code is licensed under the license found in the
    # LICENSE file in the root directory of this source tree.
    # author: adefossez

FUNCTIONS
    add_model_flags(parser)
    
    dns48(pretrained=True)
    
    dns64(pretrained=True)
    
    get_model(args)
        Load local model package or torchhub pre-trained model.
    
    master64(pretrained=True)
    
    valentini_nc(pretrained=True)

DATA
    DNS_48_URL = 'https://dl.fbaipublicfiles.com/adiyoss/denoiser/dns48-11...
    DNS_64_URL = 'https://dl.fbaipublicfiles.com/adiyoss/denoiser/dns64-a7...
    MASTER_64_URL = 'https://dl.fbaipublicfiles.com/adiyoss/denoiser/maste...
    ROOT = 'https://dl.fbaipublicfiles.com/adiyoss/denoiser/'
    VALENTINI_NC = 'https://dl.fbaipublicfiles.com/adiyoss/denoiser/valent...
    logger = <Logger de

In [89]:
print(pretrained.dns64().sample_rate)
print(pretrained.dns64().chin)

16000
1


In [90]:
help(convert_audio)

Help on function convert_audio in module denoiser.dsp:

convert_audio(wav, from_samplerate, to_samplerate, channels)
    Convert audio from a given samplerate to a target one and target number of channels.



In [91]:
#Son vocal
son1_bt=convert_audio(torch.from_numpy(son1_b).float()[None, :],44100,16000,1)
model=pretrained.dns64()
son1_dt=model(son1_bt[None])
son1_d=son1_dt[0, 0].detach().cpu().numpy()
listen(son1_d,16000)

In [92]:
#Son musical
son2_bt=convert_audio(torch.from_numpy(son2_b).float()[None, :],44100,16000,1)
model=pretrained.dns64()
son2_dt=model(son2_bt[None])
son2_d=son2_dt[0, 0].detach().cpu().numpy()
listen(son2_d,16000)

#### Conclusion DEMUCS
On remarque que le debruiteur DEMUCS filtre le son vocal correctement mais produit un résultat incohérent pour le son musical car il est entrainé sur de la parole et non de la musique.

## SGMSE+